In [1]:
import numpy as np
import pandas as pd

In [2]:
loveLetters = [
    'Dear sss,\n\tThis is xxx and I Love you from since we have met, you are a trustworthy and cautious girl also an apple to my eyes.',
    'Dear ssxs,\n\tThis is xdasfxx and I Love you from since we have met, you are a trustworthy and versatile girl also an apple to my eyes.',
    'Dear dsasdf,\n\tThis is xxx and I Love you from since we have met, you are a trustworthy and cautious boy also an apple to my eyes.',
    'Dear dfjdksjf,\n\tThis is xdasfxx and I Love you from since we have met, you are a trustworthy and versatile boy also an apple to my eyes and a non articulate girl like me can show my articulation in front of you.'
]
nonLoveLetters = [
    'Dear sss,\n\tThis is xxx and I like you from since we have met, you are a trustworthy and cautious girl also an apple to my eyes.',
    'Dear ssxs,\n\tThis is xdasfxx and I like you from since we have met, you are a trustworthy and versatile girl also an apple to my eyes.',
    'Dear dsasdf,\n\tThis is xxx and I like you from since we have met, you are a trustworthy and cautious boy also an apple to my eyes.',
    'Dear dfjdksjf,\n\tThis is xdasfxx and I like you from since we have met, you are a trustworthy and versatile boy also an apple to my eyes and a non articulate girl like me can show my articulation in front of you.'
]

In [3]:
from nltk import PorterStemmer
from collections import Counter
stemmer = PorterStemmer()
a=[stemmer.stem(word) for word in loveLetters[0].split()]
a[0]

'dear'

In [4]:
q=Counter(a+a)

In [5]:
[a[0] for a in q.most_common()[:4]]

['and', 'you', 'dear', 'sss,']

In [6]:
from sklearn.model_selection import train_test_split
X = np.array(loveLetters+nonLoveLetters,dtype="object")
Y=np.array([1]*len(loveLetters)+[0]*len(nonLoveLetters))
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [7]:
from scipy.sparse import csr_matrix
from sklearn.base import TransformerMixin,BaseEstimator
class Vectorizer(TransformerMixin,BaseEstimator):
    def __init__(self,vocab_size=50):
        self.vocab_size = vocab_size
    def fit(self,X,y=None):
        self.letters = []
        self.textARR = []
        for i in X:
            self.letters.append([stemmer.stem(word) for word in i.split()])
        for i in self.letters:
            for text in (" ".join(i)).split():
                self.textARR.append(text)
        
        self.textCounter = Counter(self.textARR)
        self.vocab_ = [a[0] for a in self.textCounter.most_common()[:self.vocab_size]]
        return self
    def transform(self,X,y=None):
        Vectors=[]
        for letter in X:
            letter = [stemmer.stem(word) for word in letter.split()]
            MinVector =[]
            for VocWord in self.vocab_:
                if VocWord in letter:
                    MinVector.append(1)
                else: 
                    MinVector.append(0)
            Vectors.append(MinVector)
       ## print(Vectors)
        return np.array(Vectors)

In [8]:
vectorizer = Vectorizer()
vectorizer.fit(loveLetters+nonLoveLetters)

Vectorizer()

In [9]:
data=vectorizer.transform(X_train)

In [10]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(data,Y_train)
model.predict(vectorizer.transform(X_test))==Y_test

array([ True,  True])

In [11]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,data,Y_train,cv=3,scoring="accuracy")

In [12]:
scores.mean()

0.0

In [13]:
scores

array([0., 0., 0.])

In [14]:
from joblib import dump
dump(model,"model.joblib")

['model.joblib']

In [15]:
model.predict(vectorizer.transform(["Dear Ssdd,I am Vxx and I wanted to tell you I LOve you."]))

array([1])

In [16]:
dump(vectorizer,"vectorizer.joblib")

['vectorizer.joblib']

In [17]:
model.predict(vectorizer.transform(["Dear Ssdd,I am Vxx and I wanted to ask you that can you send me English notes."]))

array([0])